# **Step - 1**

In [0]:
from pyspark.sql import functions as F

In [0]:
sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/dev/bronze/raw/sales.csv")
)

In [0]:
sales_df.display()

In [0]:
sales_df.printSchema()

In [0]:
sales_df.count()

# **Step - 2**

In [0]:
print("Total Rows", sales_df.count())

In [0]:
print("Total Columns", len(sales_df.columns))

In [0]:
sales_df.columns

In [0]:
sales_df.select("customer_id").distinct().count()

In [0]:
sales_df.select("product_id").distinct().count()

In [0]:
sales_df.describe().show()

# **Step - 3**

In [0]:
sales_clean_df = sales_df.dropDuplicates()

In [0]:
print("Before:", sales_df.count())
print("After:", sales_clean_df.count())

In [0]:
sales_clean_df = sales_clean_df.dropDuplicates(["transaction_id"])

In [0]:
sales_clean_df = sales_clean_df.dropna()

In [0]:
sales_clean_df.display()

# **Step 4 — Transform the Data**

In [0]:
sales_clean_df = sales_clean_df.withColumnRenamed("order_id", "Order_id").withColumnRenamed("customer_id", "Customer_id")

In [0]:
sales_clean_df = sales_clean_df.withColumn(
    "order_year",
    F.year("order_date")
)

In [0]:
sales_clean_df = sales_clean_df.withColumn(
    "order_month",
    F.month("order_date")
)

In [0]:
sales_clean_df = sales_clean_df.withColumn(
    "order_month_name",
    F.date_format("order_date", "MMMM")
)

In [0]:
sales_clean_df = sales_clean_df.withColumn(
    "order_quarter",
    F.quarter("order_date")
)

In [0]:
sales_clean_df = sales_clean_df.withColumn(
    "net_amount",
    F.col("total_amount") - F.col("discount_amount")
)

In [0]:
sales_clean_df = sales_clean_df.withColumn(
    "order_date",
    F.to_date("order_date")
)

In [0]:
sales_clean_df = sales_clean_df.withColumn(
    "load_timestamp",
    F.current_timestamp()
)

In [0]:
sales_clean_df = sales_clean_df.withColumn(
    "load_date",
    F.current_date()
)

In [0]:
sales_clean_df = sales_clean_df.withColumn(
    "file_name",
    F.col("_metadata.file_name")
)

In [0]:
sales_clean_df = sales_clean_df.withColumn(
    "file_path",
    F.col("_metadata.file_path")
)

# **Step 5 — Write the Data as Delta**

In [0]:
sales_clean_df.write.mode("overwrite").saveAsTable("dev.silver.first_sales_clean")

# **Step 6 — Analytics**

## Calculate Total Sales

In [0]:
sales_clean_df.select(
    F.round(F.sum("total_amount"), 2).alias("total_sales")
).display()

## Find Total Number of Orders

In [0]:
sales_clean_df.select(
    F.countDistinct("Order_id").alias("total_orders")
).display()

## Find Total Number of Unique Customers

In [0]:
sales_clean_df.select(
    F.countDistinct("Customer_id").alias("unique_customers")
).display()

## Calculate Total Sales for Each Customer

In [0]:
customer_sales_df = (
    sales_clean_df
    .groupBy("Customer_id")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_sales")
    )
    .orderBy(F.desc("total_sales"))
)

In [0]:
customer_sales_df.display()

## Calculate Total Sales for Each Product

In [0]:
product_sales_df = (
    sales_clean_df
    .groupBy("product_id")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_sales")
    )
    .orderBy(F.desc("total_sales"))
)

In [0]:
product_sales_df.display()

## Top 10 Customers by Sales

In [0]:
top_10_customers_df = (
    sales_clean_df
    .groupBy("Customer_id")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_sales")
    )
    .orderBy(F.desc("total_sales"))
    .limit(10)
)


In [0]:
top_10_customers_df.display()

## Top 10 Products by Sales

In [0]:
top_10_products_df = (
    sales_clean_df
    .groupBy("product_id")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_sales")
    )
    .orderBy(F.desc("total_sales"))
    .limit(10)
)

In [0]:
top_10_products_df.display()

## Find the Highest Sales Transaction

In [0]:
highest_transaction_df = (
    sales_clean_df
    .orderBy(F.desc("total_amount"))
    .limit(1)
)

In [0]:
highest_transaction_df.display()

## Find the Lowest Sales Transaction

In [0]:
lowest_transaction_df = (
    sales_clean_df
    .orderBy(F.asc("total_amount"))
    .limit(1)
)

In [0]:
lowest_transaction_df.display()

## Calculate Average Order Value

In [0]:
aov_df = sales_clean_df.select(
    (
        F.round(F.sum("total_amount") /
        F.countDistinct("Order_id"), 2)
    ).alias("average_order_value")
)

In [0]:
aov_df.display()

In [0]:
%sql
CREATE OR REPLACE VIEW dev.gold.total_amount AS
SELECT
    SUM(total_amount) AS total_sales
FROM dev.silver.first_sales_clean;

In [0]:
%sql
CREATE OR REPLACE VIEW dev.gold.total_orders_view AS
SELECT
    COUNT(DISTINCT order_id) AS total_orders
FROM dev.silver.first_sales_clean;

In [0]:
%sql
CREATE OR REPLACE VIEW dev.gold.unique_customers_view AS
SELECT
    COUNT(DISTINCT Customer_id) AS unique_customers
FROM dev.silver.first_sales_clean;

In [0]:
%sql
CREATE OR REPLACE VIEW dev.gold.customer_sales_view AS
SELECT
    Customer_id,
    SUM(total_amount) AS total_sales
FROM dev.silver.first_sales_clean
GROUP BY Customer_id;

In [0]:
%sql
CREATE OR REPLACE VIEW dev.gold.product_sales_view AS
SELECT
    product_id,
    SUM(total_amount) AS total_sales
FROM dev.silver.first_sales_clean
GROUP BY product_id;

In [0]:
%sql
CREATE OR REPLACE VIEW dev.gold.top_10_customers_view AS
SELECT
    Customer_id,
    SUM(total_amount) AS total_sales
FROM dev.silver.first_sales_clean
GROUP BY Customer_id
ORDER BY total_sales DESC
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE VIEW dev.gold.top_10_products_view AS
SELECT
    product_id,
    SUM(total_amount) AS total_sales
FROM dev.silver.first_sales_clean
GROUP BY product_id
ORDER BY total_sales DESC
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE VIEW dev.gold.highest_transaction_view AS
SELECT *
FROM dev.silver.first_sales_clean
ORDER BY total_amount DESC
LIMIT 1;

In [0]:
%sql
CREATE OR REPLACE VIEW dev.gold.lowest_transaction_view AS
SELECT *
FROM dev.silver.first_sales_clean
ORDER BY total_amount ASC
LIMIT 1;

In [0]:
%sql
CREATE OR REPLACE VIEW dev.gold.average_order_value_view AS
SELECT
    SUM(total_amount) / COUNT(DISTINCT order_id)
        AS average_order_value
FROM dev.silver.first_sales_clean;